# Module 05 — R-CNN Family

We trace the evolution from R-CNN to Fast R-CNN to Faster R-CNN,
implementing the key components along the way.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from models import RoIPool, RoIAlign, AnchorGenerator, RPNHead
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## 1. RoI Pooling

RoI Pooling extracts fixed-size features for each proposal from a shared feature map.

In [ ]:
# Create a feature map (1 batch, 256 channels, 38x50 spatial)
feature_map = torch.randn(1, 256, 38, 50)

# Three proposals: batch_idx, x1, y1, x2, y2 (in image coordinates)
rois = torch.tensor([
    [0,  50, 40, 200, 180],
    [0, 100, 20, 350, 300],
    [0, 200, 150, 580, 400],
], dtype=torch.float32)

roi_pool = RoIPool(output_size=7, spatial_scale=1/16)
pooled = roi_pool(feature_map, rois)
print('RoI Pooled shape:', pooled.shape)  # (3, 256, 7, 7)

## 2. RoI Align

RoI Align uses bilinear interpolation instead of quantisation.

In [ ]:
roi_align = RoIAlign(output_size=7, spatial_scale=1/16, sampling_ratio=2)
aligned = roi_align(feature_map, rois)
print('RoI Aligned shape:', aligned.shape)  # (3, 256, 7, 7)

# The key difference: RoI Align avoids the quantisation artifact
# that matters for precise mask prediction in Mask R-CNN

## 3. Anchor Generator

Anchors are pre-defined reference boxes placed at every feature map location.

In [ ]:
gen = AnchorGenerator(sizes=(32, 64, 128), aspect_ratios=(0.5, 1.0, 2.0))
feat = torch.randn(1, 256, 14, 14)  # 14x14 feature map
anchors = gen(feat, image_size=(224, 224))
print(f'Total anchors: {anchors.shape[0]}')  # 14*14*9 = 1764
print('First few anchors:', anchors[:3].int())

# Visualize anchors at the center location
center_idx = (14*14)//2 * 9  # middle spatial location
fig, ax = plt.subplots(figsize=(6,6))
import matplotlib.patches as mpatches
colors = ['red','blue','green','orange','purple','cyan','magenta','yellow','black']
for i, (x1,y1,x2,y2) in enumerate(anchors[center_idx:center_idx+9].int().tolist()):
    rect = mpatches.Rectangle((x1,y1),x2-x1,y2-y1,linewidth=2,edgecolor=colors[i],facecolor='none')
    ax.add_patch(rect)
ax.set_xlim(0,224); ax.set_ylim(224,0); ax.set_aspect('equal')
ax.set_title('Anchors at center location'); plt.tight_layout(); plt.show()

## 4. RPN Head

The Region Proposal Network predicts objectness and box deltas for each anchor.

In [ ]:
rpn = RPNHead(in_channels=256, num_anchors=9).to(DEVICE)
feat = torch.randn(1, 256, 14, 14).to(DEVICE)
cls_logits, bbox_deltas = rpn(feat)
print('Cls logits shape:', cls_logits.shape)   # (1, 9, 14, 14)
print('Bbox deltas shape:', bbox_deltas.shape) # (1, 36, 14, 14)

## 5. Faster R-CNN with torchvision

Let's use the pretrained torchvision implementation for inference.

In [ ]:
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights
import urllib.request, io
from PIL import Image
import torchvision.transforms.functional as TF

model = fasterrcnn_resnet50_fpn(weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT)
model.eval()

# Fetch a test image
URL = 'https://upload.wikimedia.org/wikipedia/commons/thumb/3/38/Shopping_Center_Magna_Plaza_Amsterdam_2012.jpg/640px-Shopping_Center_Magna_Plaza_Amsterdam_2012.jpg'
try:
    with urllib.request.urlopen(URL) as r: data = r.read()
    img_pil = Image.open(io.BytesIO(data)).convert('RGB')
except:
    img_pil = Image.fromarray(np.random.randint(0,255,(480,640,3), dtype=np.uint8))

img_tensor = TF.to_tensor(img_pil)
with torch.no_grad():
    preds = model([img_tensor])[0]

print('Number of detections:', len(preds['boxes']))
print('Top-5 scores:', preds['scores'][:5].round(decimals=3))

## Exercise — Box Delta Encoding/Decoding

Faster R-CNN predicts *deltas* `(dx, dy, dw, dh)` relative to anchor boxes.
Implement the encode/decode functions.

In [ ]:
### EXERCISE
def encode_boxes(anchors, gt_boxes):
    """
    Encode ground-truth boxes as (dx, dy, dw, dh) deltas relative to anchors.
    Both inputs are (N, 4) in xyxy format.
    Returns (N, 4) deltas.
    """
    # TODO
    raise NotImplementedError

def decode_boxes(anchors, deltas):
    """
    Apply deltas to anchors to get predicted boxes in xyxy format.
    anchors: (N, 4), deltas: (N, 4)
    Returns (N, 4) predicted boxes.
    """
    # TODO
    raise NotImplementedError